# 🧩 Ejemplos Reales de Clustering

En este notebook aplicaremos los algoritmos de clustering aprendidos (K-Means, Clustering Jerárquico y DBSCAN) sobre datasets reales. Analizaremos los resultados, visualizaremos los clústeres y mostraremos cómo el clustering puede ayudarnos a descubrir patrones y detectar anomalías en datos reales.

---

## 1. Importar librerías y cargar datos reales

En primer lugar, importamos las librerías necesarias y cargamos un dataset real. Usaremos el famoso dataset **Iris** y el dataset **Wine** de UCI, ambos disponibles en Scikit-Learn. Puedes adaptar el código para tus propios datos.

In [ ]:
# Importar librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score
import scipy.cluster.hierarchy as sch

sns.set_style('whitegrid')

# Cargar dataset Iris
data_iris = datasets.load_iris()
df_iris = pd.DataFrame(data_iris.data, columns=data_iris.feature_names)
df_iris['target'] = data_iris.target

# Cargar dataset Wine
data_wine = datasets.load_wine()
df_wine = pd.DataFrame(data_wine.data, columns=data_wine.feature_names)
df_wine['target'] = data_wine.target

# Mostrar primeras filas de ambos datasets
display(df_iris.head())
display(df_wine.head())

---

## 2. Preprocesamiento de datos reales

Antes de aplicar clustering, es fundamental preparar los datos: eliminar valores nulos, seleccionar variables relevantes y escalar las características. Además, usaremos PCA para reducir la dimensionalidad y facilitar la visualización.

In [ ]:
# Preprocesamiento para Iris
df_iris_clean = df_iris.dropna()
X_iris = df_iris_clean.drop('target', axis=1)
scaler_iris = StandardScaler()
X_iris_scaled = scaler_iris.fit_transform(X_iris)
# X_iris_scaled = StandardScaler().fit_transform(X_iris) 

# Preprocesamiento para Wine
df_wine_clean = df_wine.dropna()
X_wine = df_wine_clean.drop('target', axis=1)
scaler_wine = StandardScaler()
X_wine_scaled = scaler_wine.fit_transform(X_wine)

# Reducción de dimensionalidad para visualización (PCA a 2D)
pca_iris = PCA(n_components=2)
X_iris_pca = pca_iris.fit_transform(X_iris_scaled)
# X_iris_pca = PCA(n_components=2).fit_transform(X_iris_scaled)
pca_wine = PCA(n_components=2)
X_wine_pca = pca_wine.fit_transform(X_wine_scaled)

# Visualización rápida de los datos reales en 2D
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(X_iris_pca[:,0], X_iris_pca[:,1], c=df_iris_clean['target'], cmap='viridis', s=40)
axes[0].set_title('Iris (PCA 2D)')
axes[1].scatter(X_wine_pca[:,0], X_wine_pca[:,1], c=df_wine_clean['target'], cmap='tab10', s=40)
axes[1].set_title('Wine (PCA 2D)')
plt.show()

# Vamos a ver la grafica del codo
def plot_elbow(X, title):
    inertias = []
    for k in range(1, 21):
        kmeans = KMeans(n_clusters=k, random_state=42)
        kmeans.fit(X)
        inertias.append(kmeans.inertia_)
    plt.plot(range(1, 21), inertias, marker='o')
    plt.xlabel('Número de clústeres')
    plt.ylabel('Inercia')
    plt.title(f'Método del codo {title}')
    plt.show()
plot_elbow(X_iris_scaled, 'Iris')
plot_elbow(X_wine_scaled, 'Wine')

---

## 3. Clustering con K-Means en datos reales

Aplicamos K-Means a los datasets Iris y Wine. Visualizaremos los clústeres en el espacio reducido por PCA y analizaremos la calidad del agrupamiento con la métrica Silhouette Score.

In [ ]:
# K-Means en Iris
kmeans_iris = KMeans(n_clusters=2, random_state=42, n_init='auto')
labels_iris = kmeans_iris.fit_predict(X_iris_scaled)
silhouette_iris = silhouette_score(X_iris_scaled, labels_iris)

plt.figure(figsize=(6,5))
plt.scatter(X_iris_pca[:,0], X_iris_pca[:,1], c=labels_iris, cmap='Set1', s=40)
plt.title(f'Iris - K-Means (Silhouette: {silhouette_iris:.2f})')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.show()

# K-Means en Wine
kmeans_wine = KMeans(n_clusters=3, random_state=42, n_init='auto')
labels_wine = kmeans_wine.fit_predict(X_wine_scaled)
silhouette_wine = silhouette_score(X_wine_scaled, labels_wine)

plt.figure(figsize=(6,5))
plt.scatter(X_wine_pca[:,0], X_wine_pca[:,1], c=labels_wine, cmap='Set2', s=40)
plt.title(f'Wine - K-Means (Silhouette: {silhouette_wine:.2f})')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.show()

---

## 4. Clustering Jerárquico en datos reales

Ahora aplicamos el clustering jerárquico aglomerativo a los mismos datos. Visualizaremos el dendrograma y los clústeres resultantes, y los compararemos con los de K-Means.

In [ ]:
# Dendrograma para Iris
plt.figure(figsize=(10, 5))
plt.title('Dendrograma - Iris')
dend = sch.dendrogram(sch.linkage(X_iris_scaled, method='ward'))
plt.xlabel('Índice de muestra')
plt.ylabel('Distancia')
plt.show()

# Clustering Jerárquico en Iris
agg_iris = AgglomerativeClustering(n_clusters=2, linkage='ward') # Usamos 2 clusters basado en el dendrograma
labels_agg_iris = agg_iris.fit_predict(X_iris_scaled)
silhouette_agg_iris = silhouette_score(X_iris_scaled, labels_agg_iris)

plt.figure(figsize=(6,5))
plt.scatter(X_iris_pca[:,0], X_iris_pca[:,1], c=labels_agg_iris, cmap='Set1', s=40)
plt.title(f'Iris - Jerárquico (Silhouette: {silhouette_agg_iris:.2f})')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.show()

# Dendrograma para Wine
plt.figure(figsize=(10, 5))
plt.title('Dendrograma - Wine')
dend = sch.dendrogram(sch.linkage(X_wine_scaled, method='ward'))
plt.xlabel('Índice de muestra')
plt.ylabel('Distancia')
plt.show()

# Clustering Jerárquico en Wine
agg_wine = AgglomerativeClustering(n_clusters=3, linkage='ward')
labels_agg_wine = agg_wine.fit_predict(X_wine_scaled)
silhouette_agg_wine = silhouette_score(X_wine_scaled, labels_agg_wine)

plt.figure(figsize=(6,5))
plt.scatter(X_wine_pca[:,0], X_wine_pca[:,1], c=labels_agg_wine, cmap='Set2', s=40)
plt.title(f'Wine - Jerárquico (Silhouette: {silhouette_agg_wine:.2f})')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.show()

---

## 5. Clustering con DBSCAN en datos reales

Probamos DBSCAN en los mismos datos. Ajustaremos los hiperparámetros y analizaremos los puntos de ruido detectados.

In [ ]:
# DBSCAN en Iris
# Ajusta eps y min_samples según el dataset
from sklearn.neighbors import NearestNeighbors

# Curva k-distance para estimar eps en Iris
distances_iris = NearestNeighbors(n_neighbors=5).fit(X_iris_scaled).kneighbors(X_iris_scaled)[0][:, -1]
plt.figure(figsize=(6,3))
plt.plot(np.sort(distances_iris))
plt.title('Curva k-distance (Iris)')
plt.xlabel('Puntos ordenados')
plt.ylabel('Distancia al 5º vecino')
plt.grid(True)
plt.show()

# DBSCAN
dbscan_iris = DBSCAN(eps=0.85, min_samples=5)
labels_dbscan_iris = dbscan_iris.fit_predict(X_iris_scaled)
silhouette_dbscan_iris = silhouette_score(X_iris_scaled, labels_dbscan_iris, metric='euclidean') if len(set(labels_dbscan_iris)) > 1 else -1

plt.figure(figsize=(6,5))
plt.scatter(X_iris_pca[:,0], X_iris_pca[:,1], c=labels_dbscan_iris, cmap='tab10', s=40)
plt.title(f'Iris - DBSCAN (Silhouette: {silhouette_dbscan_iris:.2f})')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.show()

# DBSCAN en Wine
distances_wine = NearestNeighbors(n_neighbors=5).fit(X_wine_scaled).kneighbors(X_wine_scaled)[0][:, -1]
plt.figure(figsize=(6,3))
plt.plot(np.sort(distances_wine))
plt.title('Curva k-distance (Wine)')
plt.xlabel('Puntos ordenados')
plt.ylabel('Distancia al 5º vecino')
plt.grid(True)
plt.show()

dbscan_wine = DBSCAN(eps=3.5, min_samples=5) # Ajusta eps según la curva k-distance
labels_dbscan_wine = dbscan_wine.fit_predict(X_wine_scaled)
silhouette_dbscan_wine = silhouette_score(X_wine_scaled, labels_dbscan_wine, metric='euclidean') if len(set(labels_dbscan_wine)) > 1 else -1

plt.figure(figsize=(6,5))
plt.scatter(X_wine_pca[:,0], X_wine_pca[:,1], c=labels_dbscan_wine, cmap='tab20', s=40)
plt.title(f'Wine - DBSCAN (Silhouette: {silhouette_dbscan_wine:.2f})')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.show()

# Número de clústeres y puntos de ruido
print('Iris - DBSCAN:')
print('Nº de clústeres:', len(set(labels_dbscan_iris)) - (1 if -1 in labels_dbscan_iris else 0))
print('Nº de puntos de ruido:', np.sum(labels_dbscan_iris == -1))
print('---')
print('Wine - DBSCAN:') 
print('Nº de clústeres:', len(set(labels_dbscan_wine)) - (1 if -1 in labels_dbscan_wine else 0))
print('Nº de puntos de ruido:', np.sum(labels_dbscan_wine == -1))

---

## 6. Comparación visual y cuantitativa de los resultados

Vamos a comparar los resultados de los tres algoritmos en ambos datasets, tanto visualmente como usando métricas como el Silhouette Score y el número de clústeres encontrados.

In [ ]:
# Comparativa de resultados
resultados = pd.DataFrame({
    'Algoritmo': ['K-Means', 'Jerárquico', 'DBSCAN']*2,
    'Dataset': ['Iris']*3 + ['Wine']*3,
    'Silhouette': [silhouette_iris, silhouette_agg_iris, silhouette_dbscan_iris,
                   silhouette_wine, silhouette_agg_wine, silhouette_dbscan_wine],
    'Nº Clústeres': [len(set(labels_iris)), len(set(labels_agg_iris)), len(set(labels_dbscan_iris)) - (1 if -1 in labels_dbscan_iris else 0),
                     len(set(labels_wine)), len(set(labels_agg_wine)), len(set(labels_dbscan_wine)) - (1 if -1 in labels_dbscan_wine else 0)]
})
display(resultados)

# Visualización comparativa
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=resultados[resultados['Dataset']=='Iris'], x='Algoritmo', y='Silhouette', ax=axes[0])
axes[0].set_title('Iris - Silhouette Score')
sns.barplot(data=resultados[resultados['Dataset']=='Wine'], x='Algoritmo', y='Silhouette', ax=axes[1])
axes[1].set_title('Wine - Silhouette Score')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=resultados[resultados['Dataset']=='Iris'], x='Algoritmo', y='Nº Clústeres', ax=axes[0])
axes[0].set_title('Iris - Nº Clústeres')
sns.barplot(data=resultados[resultados['Dataset']=='Wine'], x='Algoritmo', y='Nº Clústeres', ax=axes[1])
axes[1].set_title('Wine - Nº Clústeres')
plt.show()

---

## 7. Ejemplo: Detección de anomalías con clustering

El clustering también puede ayudarnos a detectar anomalías. Los puntos etiquetados como ruido por DBSCAN suelen ser observaciones atípicas. Veamos algunos ejemplos en los datos reales.

In [ ]:
# Detección de anomalías en Iris
anomalies_iris = df_iris_clean[labels_dbscan_iris == -1]
print(f'Puntos de Iris detectados como anomalía por DBSCAN: {len(anomalies_iris)}')
display(anomalies_iris)

plt.figure(figsize=(6,5))
plt.scatter(X_iris_pca[:,0], X_iris_pca[:,1], c='lightgray', s=40, label='Normal')
plt.scatter(X_iris_pca[labels_dbscan_iris==-1,0], X_iris_pca[labels_dbscan_iris==-1,1], c='red', s=60, label='Anomalía')
plt.title('Iris - Anomalías detectadas por DBSCAN')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.legend()
plt.show()

# Detección de anomalías en Wine
anomalies_wine = df_wine_clean[labels_dbscan_wine == -1]
print(f'Puntos de Wine detectados como anomalía por DBSCAN: {len(anomalies_wine)}')
display(anomalies_wine)

plt.figure(figsize=(6,5))
plt.scatter(X_wine_pca[:,0], X_wine_pca[:,1], c='lightgray', s=40, label='Normal')
plt.scatter(X_wine_pca[labels_dbscan_wine==-1,0], X_wine_pca[labels_dbscan_wine==-1,1], c='red', s=60, label='Anomalía')
plt.title('Wine - Anomalías detectadas por DBSCAN')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.legend()
plt.show()

## Comparación de diferentes algoritmos de agrupamiento en conjuntos de datos ficticios

Este ejemplo muestra las características de diferentes algoritmos de agrupamiento en conjuntos de datos que son «interesantes», pero que siguen siendo bidimensionales. 

Con la excepción del último conjunto de datos, los parámetros de cada uno de estos pares de conjuntos de datos y algoritmos se han ajustado para producir buenos resultados de agrupamiento. 

Algunos algoritmos son más sensibles a los valores de los parámetros que otros.

El último conjunto de datos es un ejemplo de una situación «nula» para la agrupación: los datos son homogéneos y no hay una buena agrupación. 

Para este ejemplo, el conjunto de datos nulo utiliza los mismos parámetros que el conjunto de datos de la fila superior, lo que representa una discrepancia entre los valores de los parámetros y la estructura de los datos.

Si bien estos ejemplos dan una idea intuitiva de los algoritmos, es posible que esta intuición no se aplique a datos de muy alta dimensión.

In [ ]:
# Authors: The scikit-learn developers
# SPDX-License-Identifier: BSD-3-Clause

import time
import warnings
from itertools import cycle, islice

import matplotlib.pyplot as plt
import numpy as np

from sklearn import cluster, datasets, mixture
from sklearn.neighbors import kneighbors_graph
from sklearn.preprocessing import StandardScaler

# ============
# Generar dataset. Elegimos un tamaño lo suficientemente grande como para ver la escalabilidad
# de los algoritmos, pero no tan grande como para evitar tiempos de ejecución demasiado largos
# ============
n_samples = 500
seed = 30
noisy_circles = datasets.make_circles(
    n_samples=n_samples, factor=0.5, noise=0.05, random_state=seed
)
noisy_moons = datasets.make_moons(n_samples=n_samples, noise=0.05, random_state=seed)
blobs = datasets.make_blobs(n_samples=n_samples, random_state=seed)
rng = np.random.RandomState(seed)
no_structure = rng.rand(n_samples, 2), None

# Anisotropicly distributed data
random_state = 170
X, y = datasets.make_blobs(n_samples=n_samples, random_state=random_state)
transformation = [[0.6, -0.6], [-0.4, 0.8]]
X_aniso = np.dot(X, transformation)
aniso = (X_aniso, y)

# blobs with varied variances
varied = datasets.make_blobs(
    n_samples=n_samples, cluster_std=[1.0, 2.5, 0.5], random_state=random_state
)

# ============
# Set up cluster parameters
# ============
plt.figure(figsize=(9 * 2 + 3, 13))
plt.subplots_adjust(
    left=0.02, right=0.98, bottom=0.001, top=0.95, wspace=0.05, hspace=0.01
)

plot_num = 1

default_base = {
    "quantile": 0.3,
    "eps": 0.3,
    "damping": 0.9,
    "preference": -200,
    "n_neighbors": 3,
    "n_clusters": 3,
    "min_samples": 7,
    "xi": 0.05,
    "min_cluster_size": 0.1,
    "allow_single_cluster": True,
    "hdbscan_min_cluster_size": 15,
    "hdbscan_min_samples": 3,
    "random_state": 42,
}

datasets = [
    (
        noisy_circles,
        {
            "damping": 0.77,
            "preference": -240,
            "quantile": 0.2,
            "n_clusters": 2,
            "min_samples": 7,
            "xi": 0.08,
        },
    ),
    (
        noisy_moons,
        {
            "damping": 0.75,
            "preference": -220,
            "n_clusters": 2,
            "min_samples": 7,
            "xi": 0.1,
        },
    ),
    (
        varied,
        {
            "eps": 0.18,
            "n_neighbors": 2,
            "min_samples": 7,
            "xi": 0.01,
            "min_cluster_size": 0.2,
        },
    ),
    (
        aniso,
        {
            "eps": 0.15,
            "n_neighbors": 2,
            "min_samples": 7,
            "xi": 0.1,
            "min_cluster_size": 0.2,
        },
    ),
    (blobs, {"min_samples": 7, "xi": 0.1, "min_cluster_size": 0.2}),
    (no_structure, {}),
]

for i_dataset, (dataset, algo_params) in enumerate(datasets):
    # Actualizar parámetros con los específicos del dataset
    params = default_base.copy()
    params.update(algo_params)

    X, y = dataset

    # normalizar el dataset para facilitar la selección de parámetros
    X = StandardScaler().fit_transform(X)

    # estimar el bandwidth para mean shift
    bandwidth = cluster.estimate_bandwidth(X, quantile=params["quantile"])

    # conectividad para structured Ward
    connectivity = kneighbors_graph(
        X, n_neighbors=params["n_neighbors"], include_self=False
    )
    # hacer simétrica la matriz de conectividad
    connectivity = 0.5 * (connectivity + connectivity.T)

    # ============
    # Crear los algoritmos de clustering
    # ============
    ms = cluster.MeanShift(bandwidth=bandwidth, bin_seeding=True)
    two_means = cluster.MiniBatchKMeans(
        n_clusters=params["n_clusters"],
        random_state=params["random_state"],
    )
    ward = cluster.AgglomerativeClustering(
        n_clusters=params["n_clusters"], linkage="ward", connectivity=connectivity
    )
    spectral = cluster.SpectralClustering(
        n_clusters=params["n_clusters"],
        eigen_solver="arpack",
        affinity="nearest_neighbors",
        random_state=params["random_state"],
    )
    dbscan = cluster.DBSCAN(eps=params["eps"])
    hdbscan = cluster.HDBSCAN(
        min_samples=params["hdbscan_min_samples"],
        min_cluster_size=params["hdbscan_min_cluster_size"],
        allow_single_cluster=params["allow_single_cluster"],
    )
    optics = cluster.OPTICS(
        min_samples=params["min_samples"],
        xi=params["xi"],
        min_cluster_size=params["min_cluster_size"],
    )
    affinity_propagation = cluster.AffinityPropagation(
        damping=params["damping"],
        preference=params["preference"],
        random_state=params["random_state"],
    )
    average_linkage = cluster.AgglomerativeClustering(
        linkage="average",
        metric="cityblock",
        n_clusters=params["n_clusters"],
        connectivity=connectivity,
    )
    birch = cluster.Birch(n_clusters=params["n_clusters"])
    gmm = mixture.GaussianMixture(
        n_components=params["n_clusters"],
        covariance_type="full",
        random_state=params["random_state"],
    )

    clustering_algorithms = (
        ("MiniBatch\nKMeans", two_means),
        ("Affinity\nPropagation", affinity_propagation),
        ("MeanShift", ms),
        ("Spectral\nClustering", spectral),
        ("Ward", ward),
        ("Agglomerative\nClustering", average_linkage),
        ("DBSCAN", dbscan),
        ("HDBSCAN", hdbscan),
        ("OPTICS", optics),
        ("BIRCH", birch),
        ("Gaussian\nMixture", gmm),
    )

    for name, algorithm in clustering_algorithms:
        t0 = time.time()

        # catch warnings related to kneighbors_graph
        with warnings.catch_warnings():
            warnings.filterwarnings(
                "ignore",
                message="the number of connected components of the "
                "connectivity matrix is [0-9]{1,2}"
                " > 1. Completing it to avoid stopping the tree early.",
                category=UserWarning,
            )
            warnings.filterwarnings(
                "ignore",
                message="Graph is not fully connected, spectral embedding"
                " may not work as expected.",
                category=UserWarning,
            )
            algorithm.fit(X)

        t1 = time.time()
        if hasattr(algorithm, "labels_"):
            y_pred = algorithm.labels_.astype(int)
        else:
            y_pred = algorithm.predict(X)

        plt.subplot(len(datasets), len(clustering_algorithms), plot_num)
        if i_dataset == 0:
            plt.title(name, size=18)

        colors = np.array(
            list(
                islice(
                    cycle(
                        [
                            "#377eb8",
                            "#ff7f00",
                            "#4daf4a",
                            "#f781bf",
                            "#a65628",
                            "#984ea3",
                            "#999999",
                            "#e41a1c",
                            "#dede00",
                        ]
                    ),
                    int(max(y_pred) + 1),
                )
            )
        )
        # add black color for outliers (if any)
        colors = np.append(colors, ["#000000"])
        plt.scatter(X[:, 0], X[:, 1], s=10, color=colors[y_pred])

        plt.xlim(-2.5, 2.5)
        plt.ylim(-2.5, 2.5)
        plt.xticks(())
        plt.yticks(())
        plt.text(
            0.99,
            0.01,
            ("%.2fs" % (t1 - t0)).lstrip("0"),
            transform=plt.gca().transAxes,
            size=15,
            horizontalalignment="right",
        )
        plot_num += 1

plt.show()
print("Este ejemplo visualiza varios algoritmos de clustering aplicados a diferentes datasets sintéticos.")
print("Nos puede ayudar a entender cómo funcionan y comparar su rendimiento visualmente.")